In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## مرحله ۱: تعریف مدل‌های Pydantic برای خروجی‌های ساخت‌یافته

این مدل‌ها **اسکما** را تعریف می‌کنند که عامل‌ها بازمی‌گردانند. استفاده از `response_format` همراه با Pydantic تضمین می‌کند:
- ✅ استخراج داده با نوع امن
- ✅ اعتبارسنجی خودکار
- ✅ بدون خطاهای تجزیه از پاسخ‌های متنی آزاد
- ✅ مسیریابی شرطی آسان بر اساس فیلدها


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## مرحله ۲: ایجاد ابزار رزرو هتل

این ابزاری است که **availability_agent** برای بررسی در دسترس بودن اتاق‌ها فراخوانی خواهد کرد. ما از دکوراتور `@ai_function` استفاده می‌کنیم تا:
- تبدیل یک تابع پایتون به ابزاری قابل فراخوانی توسط هوش مصنوعی
- ایجاد خودکار طرح‌واره JSON برای LLM
- مدیریت اعتبارسنجی پارامترها
- فعال‌سازی فراخوانی خودکار توسط عوامل

برای این دموی:
- **استکهلم، سیاتل، توکیو، لندن، آمستردام** → اتاق‌های موجود ✅
- **سایر شهرها** → بدون اتاق ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## گام ۳: تعریف توابع شرط برای مسیر‌دهی

این توابع پاسخ عامل را بررسی می‌کنند و مشخص می‌کنند که کدام مسیر در جریان کار دنبال شود.

**الگوی کلیدی:**
۱. بررسی کنید که پیام از نوع `AgentExecutorResponse` است
۲. خروجی ساختاریافته (مدل پایدنتیک) را تحلیل کنید
۳. مقدار `True` یا `False` را برای کنترل مسیر برگردانید

جریان کار این شرایط را روی **لبه‌ها** ارزیابی می‌کند تا تصمیم بگیرد کدام اجراکننده بعدی فراخوانی شود.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## مرحله ۴: ایجاد اجرایی‌کننده نمایش سفارشی

اجرایی‌کننده‌ها اجزای جریان کاری هستند که تبدیل‌ها یا اثرات جانبی را انجام می‌دهند. ما از دکوراتور `@executor` برای ایجاد یک اجرایی‌کننده سفارشی که نتیجه نهایی را نمایش می‌دهد استفاده می‌کنیم.

**مفاهیم کلیدی:**
- `@executor(id="...")` - ثبت یک تابع به عنوان اجرایی‌کننده جریان کاری
- `WorkflowContext[Never, str]` - راهنمای نوع برای ورودی/خروجی
- `ctx.yield_output(...)` - تولید نتیجه نهایی جریان کاری


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## مرحله ۵: بارگذاری متغیرهای محیطی

کلاینت LLM را پیکربندی کنید. این مثال با موارد زیر کار می‌کند:
- **Microsoft Foundry** / **Azure OpenAI** (API پاسخ‌ها — توصیه شده)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## مرحله ۶: ایجاد عوامل هوش مصنوعی با خروجی‌های ساختاریافته

ما **سه عامل تخصصی** ایجاد می‌کنیم که هر کدام در یک `AgentExecutor` پیچیده شده‌اند:

۱. **availability_agent** - بررسی در دسترس بودن هتل با استفاده از ابزار
۲. **alternative_agent** - پیشنهاد شهرهای جایگزین (وقتی اتاقی نیست)
۳. **booking_agent** - تشویق به رزرو (وقتی اتاق‌ها موجودند)

**ویژگی‌های کلیدی:**
- `tools=[hotel_booking]` - ابزار را به عامل می‌دهد
- `response_format=PydanticModel` - خروجی JSON ساختاریافته اجباری می‌کند
- `AgentExecutor(..., id="...")` - عامل را برای استفاده در جریان کاری پیچیده می‌کند


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## گام ۷: ساخت جریان کاری با یال‌های شرطی

اکنون از `WorkflowBuilder` برای ساخت گراف با مسیر‌یابی شرطی استفاده می‌کنیم:

**ساختار جریان کاری:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**روش‌های کلیدی:**
- `.set_start_executor(...)` - تعیین نقطه ورود
- `.add_edge(from, to, condition=...)` - افزودن یال شرطی
- `.build()` - نهایی کردن جریان کاری


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## گام ۸: اجرای تست شماره ۱ - شهر بدون دسترس‌پذیری (پاریس)

بیایید مسیر **بدون دسترس‌پذیری** را با درخواست هتل‌ها در پاریس (که در شبیه‌سازی ما اتاقی ندارد) تست کنیم.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## مرحله ۹: اجرای تست حالت ۲ - شهر با موجودی (استکهلم)

حالا بیایید مسیر **موجودی** را با درخواست هتل‌ها در استکهلم (که اتاق‌هایی در شبیه‌سازی ما دارد) آزمایش کنیم.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## نکات کلیدی و گام‌های بعدی

### ✅ آنچه آموخته‌اید:

1. **الگوی WorkflowBuilder**
   - از `.set_start_executor()` برای تعریف نقطه شروع استفاده کنید
   - از `.add_edge(from, to, condition=...)` برای مسیریابی شرطی بهره ببرید
   - برای نهایی کردن روند کار، `.build()` را فراخوانی کنید

2. **مسیریابی شرطی**
   - توابع شرطی پاسخ `AgentExecutorResponse` را بررسی می‌کنند
   - نتایج ساختاریافته را برای تصمیم‌گیری در مسیریابی تجزیه می‌کنند
   - با بازگرداندن `True` یک مسیر فعال می‌شود، با `False` مسیر عبور داده می‌شود

3. **ادغام ابزارها**
   - از `@ai_function` برای تبدیل توابع پایتون به ابزارهای هوش مصنوعی استفاده کنید
   - عوامل (Agents) در صورت نیاز به طور خودکار از ابزارها استفاده می‌کنند
   - ابزارها JSON برمی‌گردانند که عوامل قادر به تجزیه آن هستند

4. **نتایج ساختاریافته**
   - از مدل‌های Pydantic برای استخراج داده‌های نوع-ایمن استفاده کنید
   - هنگام ایجاد عوامل، `response_format=MyModel` را تنظیم کنید
   - پاسخ‌ها را با `Model.model_validate_json()` تجزیه کنید

5. **اجراکننده‌های سفارشی**
   - برای ایجاد اجزای روند کار از `@executor(id="...")` استفاده کنید
   - اجراکننده‌ها می‌توانند داده‌ها را تبدیل یا اثرات جانبی انجام دهند
   - برای تولید نتایج روند کار، از `ctx.yield_output()` بهره ببرید

### 🚀 کاربردهای واقعی:

- **رزرو سفر**: بررسی در دسترس بودن، ارائه جایگزین‌ها، مقایسه گزینه‌ها
- **خدمات مشتریان**: مسیریابی بر اساس نوع مشکل، احساسات، اولویت
- **تجارت الکترونیک**: بررسی موجودی، پیشنهاد جایگزین‌ها، پردازش سفارش‌ها
- **اعتدال محتوا**: مسیریابی بر اساس نمرات سمّیت، پرچم‌های کاربران
- **روندهای تأیید**: مسیریابی بر اساس مبلغ، نقش کاربر، سطح ریسک
- **پردازش چند مرحله‌ای**: مسیریابی بر اساس کیفیت داده‌ها، کامل بودن

### 📚 گام‌های بعدی:

- اضافه کردن شرایط پیچیده‌تر (معیارهای چندگانه)
- پیاده‌سازی حلقه‌ها با مدیریت وضعیت روند کار
- افزودن زیرروندها برای اجزای قابل استفاده مجدد
- ‌ادغام با APIهای واقعی (رزرو هتل، سیستم‌های موجودی)
- افزودن مدیریت خطا و مسیرهای جایگزین
- تجسم روندهای کار با ابزارهای تجسم ساخته‌شده


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
